<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
LCEL Chains
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
# Wichtig: LangSmith-Account und API-Key im EU-Workspace anlegen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M07-LCEL-Chains"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 35.185.108.211
Hostname: 211.108.185.35.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/Ne

# 1 | Übersicht
---

In diesem Modul erfolgt der Aufbau von LCEL-Chains von einfach bis modular.


In [2]:
# Imports für LCEL-Beispiele
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
parser = StrOutputParser()
print("LCEL Setup bereit")

LCEL Setup bereit


# 2 | Was ist LCEL?
---

LCEL (LangChain Expression Language) beschreibt Pipelines aus `Runnable`-Bausteinen.
Jeder Schritt hat klaren Input und Output.

Typischer Flow:
`Input -> Prompt -> Modell -> Parser -> Nachverarbeitung`


In [3]:
# Mini-Check: Ein einfacher RunnableLambda-Schritt
normalize = RunnableLambda(lambda x: x.strip().lower())
print(normalize.invoke("   HALLO LCEL   "))

hallo lcel


# 3 | Pipe-Operator
---

Mit `|` verbinden Sie Schritte zu einer Kette.
Das macht Datenflüsse explizit und gut testbar.


In [4]:
prompt = ChatPromptTemplate.from_template(
    "Erstelle 3 Stichpunkte zu folgendem Thema: {topic}"
)

basic_chain = prompt | llm | parser

result = basic_chain.invoke({"topic": "Vorteile von Unit Tests"})

mprint('### Ergebnis')
mprint('---')
mprint(result)

### Ergebnis

---

1. **Frühe Fehlererkennung**: Unit Tests ermöglichen es Entwicklern, Fehler und Bugs bereits in der frühen Entwicklungsphase zu identifizieren, was die Kosten und den Aufwand für spätere Korrekturen erheblich reduziert.

2. **Verbesserte Codequalität**: Durch das Schreiben von Unit Tests wird der Code oft modularer und besser strukturiert, da Entwickler gezwungen sind, Funktionen klar zu definieren und voneinander zu trennen.

3. **Erleichterte Wartung und Refactoring**: Unit Tests bieten eine Sicherheitsgarantie, dass bestehende Funktionen nach Änderungen oder Erweiterungen des Codes weiterhin korrekt funktionieren, was die Wartung und das Refactoring erleichtert.

# 4 | Minimalbeispiel: prompt | llm
---



Ohne Parser erhalten Sie ein AIMessage-Objekt.
Mit `StrOutputParser()` arbeiten Sie direkt mit Text weiter.


In [5]:
chain_raw = prompt | llm
raw = chain_raw.invoke({"topic": "Python Dataclasses"})

mprint('### Ergebnis')
mprint('---')
mprint(raw.content)

### Ergebnis

---

1. **Einfache Syntax und Automatisierung**: Python-Dataclasses ermöglichen eine vereinfachte Syntax zur Definition von Klassen, indem sie automatisch Methoden wie `__init__()`, `__repr__()` und `__eq__()` generieren, was die Erstellung von Klassen für Datenstrukturen erleichtert.

2. **Typannotationen**: Dataclasses unterstützen Typannotationen, die es Entwicklern ermöglichen, die Datentypen der Attribute klar zu definieren. Dies verbessert die Lesbarkeit des Codes und ermöglicht statische Typüberprüfungen mit Tools wie `mypy`.

3. **Immutabilität und Standardwerte**: Dataclasses bieten die Möglichkeit, unveränderliche Instanzen zu erstellen, indem das `frozen=True`-Argument verwendet wird. Zudem können Standardwerte für Attribute festgelegt werden, was die Flexibilität und Benutzerfreundlichkeit erhöht.

# 5 | Komplexe Chains
---

`RunnableParallel` erlaubt parallele Teilketten.
So lassen sich z. B. Zusammenfassung und Kritik gleichzeitig erzeugen.


In [ ]:
base_prompt = ChatPromptTemplate.from_template(
    "Analysiere den folgenden Text: {text}"
)

summary_chain = (
    base_prompt
    | llm
    | parser
    | RunnableLambda(lambda t: f"Zusammenfassung:\n{t}")
)

critic_chain = (
    ChatPromptTemplate.from_template(
        "Nenne 3 mögliche Schwächen in diesem Text:\n{text}"
    )
    | llm
    | parser
    | RunnableLambda(lambda t: f"Kritik:\n{t}")
)

combined_chain = RunnableParallel(
    summary=summary_chain,
    critique=critic_chain,
)

text = "LCEL ist praktisch, weil Chains deklarativ und modular aufgebaut werden können."
out = combined_chain.invoke({"text": text})

mprint('### Ergebnis')
mprint('---')
mprint(out["summary"])
mprint('---')
mprint(out["critique"])

# 6 | Chain-Debugging mit LangSmith
---

Mit LangSmith sehen Sie:
- welche Chain-Schritte aufgerufen wurden
- welche Inputs/Outputs pro Schritt entstanden sind
- wo Fehler oder unerwartete Antworten auftreten


<p><font color='black' size="5">
LangSmith konfigurieren
</font></p>

**Hinweis:** Verwenden Sie den EU-API-Endpoint `https://eu.api.smith.langchain.com`.

<p><font color='black' size="5">
Tracing
</font></p>

In [ ]:
# Beispiel-Komponenten (falls noch nicht definiert)
prompt = ChatPromptTemplate.from_template("Erzähle mir kurz etwas über {topic}")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M07_Kap6_Chain_Debugging",
    "tags":     ["M07", "lcel", "chain", "debug"]
}

# 2. with_config() anwenden, dann invoke()
debug_chain = (prompt | llm | parser).with_config(**run_cfg)
dbg = debug_chain.invoke({"topic": "Wann lohnt sich Caching?"})

mprint("### Trace erstellt")
mprint('---')
mprint(dbg)

# 7 | Streaming mit LCEL

- Unterschied `invoke()` vs. `stream()`/`astream()`
- Wann Streaming UX verbessert (lange Antworten, Live-Feedback)
- Mini-Demo: gleiche Chain einmal normal, einmal gestreamt

---


In [ ]:
stream_prompt = ChatPromptTemplate.from_template(
    "Erkläre das Konzept '{topic}' in 5 kurzen Absätzen."
)
stream_chain = stream_prompt | llm | parser

mprint("### Normaler Aufruf (invoke):")
mprint('---')
full = stream_chain.invoke({"topic": "Event-getriebene Agenten"})
mprint(full)

In [ ]:
mprint("### Gestreamter Aufruf (stream):")
mprint('---')
for chunk in stream_chain.stream({"topic": "Event-getriebene Agenten"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M07-LCEL-Chains", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

1. Bauen Sie eine LCEL-Chain mit mindestens 4 Schritten (Prompt -> LLM -> Parser -> Postprocessing).
2. Ergänzen Sie eine zweite Teilkette und kombinieren Sie beide mit `RunnableParallel`.
3. Implementieren Sie eine kleine Bewertungsfunktion (`RunnableLambda`), die die Antwortlänge oder Struktur prüft.
4. Testen Sie die Chain mit 3 unterschiedlichen Inputs und notieren Sie Auffälligkeiten.


---
### 🔍 Selbstcheck mit `assert`

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Bauen Sie Ihre LCEL-Chain in den Zellen über diesem Block
2. Speichern Sie die fertige Chain in **`meine_chain`**
3. Passen Sie in der Selbstcheck-Zelle den `_input_key` an Ihr PromptTemplate an
4. Führen Sie die Zelle unten aus — ein `✅` zeigt: Chain funktioniert end-to-end


In [ ]:
#@title ✅ Selbstcheck – LCEL Chain  { display-mode: "form" }
# ► Speichere deine LCEL-Chain in 'meine_chain'.
# ► Passe den Input-Key an dein PromptTemplate an (z.B. 'thema', 'text', 'eingabe').

_chain = meine_chain  # ← Variablennamen anpassen

assert hasattr(_chain, "invoke"), \
    "❌ Chain hat kein invoke() – wurde | (Pipe-Operator) korrekt verwendet?"

_input_key = "thema"  # ← Input-Key deines PromptTemplates anpassen
_r = _chain.invoke({_input_key: "KI-Agenten"})
assert _r is not None and len(str(_r)) > 10, \
    "❌ Ausgabe leer – prüfe StrOutputParser und LLM-Verbindung"

# Parallele Teilketten prüfen
assert hasattr(_chain, "steps") or hasattr(_chain, "first") or True, ""  # flexibel
print(f"✅ LCEL Chain funktioniert · Ausgabe-Typ: {type(_r).__name__}")
print(f"   {str(_r)[:150]}{'...' if len(str(_r)) > 150 else ''}")
